# 03. 과적합과 일반화 — 함정을 직접 재현하기

**대응 강의:** [4강 모델의 일반화](../../Lecture/03-머신러닝-기본개념/04-모델의-일반화.md)

이 노트북에서 배우는 것:
- 다항식 차수를 올려가며 **과소적합 → 적절 → 과적합**으로 변하는 모습을 눈으로 본다
- 훈련 점수와 테스트 점수의 **격차**로 과적합을 진단한다
- **데이터 누수(data leakage)** 가 점수를 어떻게 부풀리는지 재현한다

> '훈련을 잘 맞히는 것 ≠ 일반화'를 몸으로 느끼는 노트북입니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error

# 숨은 진짜 관계: 부드러운 곡선 (cos). 거기에 노이즈를 섞는다.
rng = np.random.default_rng(0)
def true_fn(x):
    return np.cos(1.5 * np.pi * x)

n = 30
X = np.sort(rng.uniform(0, 1, n))
y = true_fn(X) + rng.normal(0, 0.15, n)
X = X.reshape(-1, 1)
print('샘플 수:', n)

## 1. 차수에 따른 과소적합 → 적절 → 과적합

차수 1(직선)은 너무 단순(과소적합), 차수 15는 노이즈까지 외움(과적합), 그 사이에 적절한 지점이 있습니다.

In [ ]:
degrees = [1, 4, 15]
titles = ['과소적합 (너무 단순)', '적절 (good fit)', '과적합 (노이즈 암기)']
x_plot = np.linspace(0, 1, 200).reshape(-1, 1)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, d, t in zip(axes, degrees, titles):
    model = make_pipeline(PolynomialFeatures(d), LinearRegression())
    model.fit(X, y)
    ax.scatter(X, y, color='k', s=20, label='데이터')
    ax.plot(x_plot, true_fn(x_plot), 'g--', alpha=0.6, label='진짜 함수')
    ax.plot(x_plot, model.predict(x_plot), 'r-', label=f'차수 {d} 모델')
    ax.set_ylim(-2, 2); ax.set_title(t); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 2. 훈련 오차 vs 테스트 오차로 과적합 진단

차수를 1부터 15까지 올리며, **훈련 오차**와 **테스트 오차**를 각각 그려봅니다.

- 훈련 오차는 차수가 올라갈수록 계속 줄어듭니다 (외우니까).
- 하지만 테스트 오차는 어느 지점부터 **다시 올라갑니다** → 그 지점이 과적합 시작!

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=1)

max_deg = 15
train_err, test_err = [], []
for d in range(1, max_deg + 1):
    m = make_pipeline(PolynomialFeatures(d), LinearRegression())
    m.fit(X_tr, y_tr)
    train_err.append(mean_squared_error(y_tr, m.predict(X_tr)))
    test_err.append(mean_squared_error(y_te, m.predict(X_te)))

plt.figure(figsize=(8, 5))
plt.plot(range(1, max_deg + 1), train_err, 'o-', label='훈련 오차')
plt.plot(range(1, max_deg + 1), test_err, 's-', label='테스트 오차')
plt.yscale('log')
plt.axvspan(8, max_deg, alpha=0.1, color='red')
plt.text(11, max(test_err), '과적합 구간', color='red')
plt.xlabel('다항식 차수 (모델 복잡도)'); plt.ylabel('오차 (MSE, 로그)')
plt.title('복잡도가 올라가면 훈련은 좋아져도 테스트는 망가진다')
plt.legend(); plt.show()

best = int(np.argmin(test_err)) + 1
print('테스트 오차가 가장 낮은 차수(적절 복잡도):', best)

## 3. 데이터 누수(Data Leakage) 재현 — 점수를 부풀리는 함정

스케일링을 **분할 전에** 전체 데이터로 하면, 훈련이 테스트 통계를 미리 엿봅니다.
올바른 방법(분할 후, 훈련 데이터로만 fit)과 점수를 비교합니다.

> 차이를 도드라지게 보려고 일부러 샘플이 적은 고차원 상황을 만듭니다.

In [ ]:
from sklearn.linear_model import Ridge

# 타깃과 무관한 순수 노이즈 특성들 (정상이라면 성능이 형편없어야 정상)
rng2 = np.random.default_rng(42)
n_samples, n_features = 60, 200
Xn = rng2.normal(size=(n_samples, n_features))
yn = rng2.normal(size=n_samples)   # 특성과 아무 관계 없는 타깃!

# ❌ 잘못된 방법: 전체 데이터로 스케일링한 뒤 분할 (테스트 정보 누수)
Xn_leak = StandardScaler().fit_transform(Xn)
Xtr, Xte, ytr, yte = train_test_split(Xn_leak, yn, test_size=0.3, random_state=0)
leak_model = Ridge().fit(Xtr, ytr)
leak_score = leak_model.score(Xte, yte)

# ✅ 올바른 방법: 분할 먼저 → 훈련 데이터로만 스케일러 fit
Xtr2, Xte2, ytr2, yte2 = train_test_split(Xn, yn, test_size=0.3, random_state=0)
sc = StandardScaler().fit(Xtr2)          # 훈련 데이터로만 학습
clean_model = Ridge().fit(sc.transform(Xtr2), ytr2)
clean_score = clean_model.score(sc.transform(Xte2), yte2)

print('타깃은 특성과 무관 → 제대로 했다면 R^2는 0 근처(혹은 음수)여야 정상\n')
print(f'❌ 누수 버전 테스트 R^2 : {leak_score:.3f}  <- 비현실적으로 높으면 누수 의심!')
print(f'✅ 정상 버전 테스트 R^2 : {clean_score:.3f}  <- 현실(관계 없으니 낮음)')

> 💡 위에서 누수 버전 점수가 정상 버전보다 부자연스럽게 높게 나옵니다.
> 실전에서 **"점수가 너무 좋으면"** 기뻐하기 전에 누수부터 의심하세요.
>
> **예방책:** scikit-learn `Pipeline`에 스케일러를 넣으면, 교차검증 시 각 분할의
> 훈련 데이터로만 자동으로 fit 되어 누수가 구조적으로 차단됩니다.

## 정리

- 모델이 복잡할수록 훈련 오차는 줄지만, 테스트 오차는 어느 순간부터 늘어난다(과적합).
- 과적합 진단 = **훈련 점수와 테스트 점수의 격차**.
- 데이터 누수는 점수를 비현실적으로 부풀린다. **분할을 항상 가장 먼저.**

## 🎯 직접 해보기

1. Part 1에서 차수 리스트를 `[2, 6, 25]`로 바꿔 그림이 어떻게 변하는지 보세요.
2. Part 2에서 노이즈 크기(`0.15`)를 `0.4`로 키우면 '적절 차수'가 어디로 이동하나요?
3. Part 3을 `make_pipeline(StandardScaler(), Ridge())` + `cross_val_score`로 다시 짜서, 파이프라인이 누수를 막아주는지 확인해 보세요.